Training protocol: use CTC silver segmentation for supervision, as in the ELEPHANT paper (https://elifesciences.org/articles/69380). This project trains its own binary 3D U-Net from scratch. Recording 01 is used for training; recording 02 is reserved for a localization diagnostic against partial tracking annotations.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--only-binary=:all:', 'imagecodecs'], check=True)

import hashlib, json, re, urllib.request, zipfile
from pathlib import Path
import numpy as np
import tifffile
import torch
from torch import nn
from torch.nn import functional as F
from scipy import ndimage as ndi
from scipy.spatial import cKDTree

assert torch.cuda.is_available(), 'Enable Kaggle GPU'
torch.manual_seed(42)
rng = np.random.default_rng(42)
archive = Path('/tmp/ce.zip')
url = 'https://data.celltrackingchallenge.net/training-datasets/Fluo-N3DH-CE.zip'
urllib.request.urlretrieve(url, archive)
with archive.open('rb') as f:
    digest = hashlib.file_digest(f, 'sha256').hexdigest()
out = Path('/kaggle/working')
with zipfile.ZipFile(archive) as z:
    def files(pattern):
        return {int(m[1]): name for name in z.namelist() if (m := re.search(pattern, name))}
    raw = files(r'(?:^|/)01/t(\d+)\.tif$')
    silver = files(r'(?:^|/)01_ST/SEG/man_seg(\d+)\.tif$')
    test = files(r'(?:^|/)02/t(\d+)\.tif$')
    truth = files(r'(?:^|/)02_GT/TRA/man_track(\d+)\.tif$')
    train_ids, test_ids = sorted(raw.keys() & silver.keys()), sorted(test.keys() & truth.keys())
    assert train_ids and test_ids, 'Missing full-volume silver training masks or held-out tracking labels'
    def read(name):
        with z.open(name) as f:
            import io
            return tifffile.imread(io.BytesIO(f.read()))
    def normalize(a):
        lo, hi = np.percentile(a, (1, 99.8))
        return np.clip((a.astype(np.float32)-lo)/max(1,hi-lo),0,1)
    def block(a,b):
        return nn.Sequential(nn.Conv3d(a,b,3,padding=1),nn.GroupNorm(4,b),nn.SiLU(),
                             nn.Conv3d(b,b,3,padding=1),nn.GroupNorm(4,b),nn.SiLU())
    class Detector(nn.Module):
        def __init__(self):
            super().__init__()
            self.enc, self.deep, self.dec, self.head = block(1,8), block(8,16), block(24,8), nn.Conv3d(8,1,1)
        def forward(self,x):
            a=self.enc(x)
            b=self.deep(F.avg_pool3d(a,(1,2,2)))
            return self.head(self.dec(torch.cat([a,F.interpolate(b,size=a.shape[2:],mode='trilinear',align_corners=False)],1)))
    model=Detector().cuda()
    opt=torch.optim.AdamW(model.parameters(),lr=0.001)
    updates=0
    for epoch in range(1,11):
        model.train()
        losses=[]
        for t in rng.permutation(train_ids):
            a=normalize(read(raw[t]))
            labels=read(silver[t])
            assert a.shape==labels.shape and a.ndim==3
            positive=np.argwhere(labels>0)
            if not len(positive):
                continue
            size=np.minimum(a.shape,(16,64,64))
            for _ in range(2):
                center=positive[rng.integers(len(positive))]
                start=np.clip(center-size//2+rng.integers(-6,7,size=3),0,np.array(a.shape)-size)
                slices=tuple(slice(int(s),int(s+n)) for s,n in zip(start,size))
                x=torch.as_tensor(a[slices].copy(), dtype=torch.float32, device='cuda')[None,None]
                y=torch.from_numpy((labels[slices]>0).astype(np.float32))[None,None].cuda()
                logits=model(x)
                loss=F.binary_cross_entropy_with_logits(logits,y)
                assert torch.isfinite(loss)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
                updates+=1
                losses.append(float(loss.detach()))
        assert losses
        print(json.dumps(dict(epoch=epoch,optimizer_steps=updates,loss=float(np.mean(losses)))),flush=True)
        torch.save(dict(model=model.state_dict(),epoch=epoch,optimizer_steps=updates),out/'detector.pt')
    model.eval()
    report=[]
    with torch.inference_mode():
        for t in test_ids:
            a=normalize(read(test[t]))
            labels=read(truth[t])
            assert a.shape==labels.shape
            # Overlap depth tiles and discard interior tile margins.
            scores=np.empty_like(a)
            for start in range(0,len(a),8):
                lo,hi=max(0,start-4),min(len(a),start+12)
                x=torch.as_tensor(a[lo:hi], dtype=torch.float32, device='cuda')[None,None]
                end=min(len(a),start+8)
                scores[start:end]=model(x).sigmoid()[0,0,start-lo:end-lo].cpu().numpy()
            components,n=ndi.label(scores>=0.5)
            predicted=np.asarray(ndi.center_of_mass(scores,components,range(1,n+1))).reshape(-1,3)
            ids=np.unique(labels)
            actual=np.asarray(ndi.center_of_mass(labels>0,labels,ids[ids>0])).reshape(-1,3)
            # Partial annotations: report nearest-prediction distances without calling unmatched predictions false positives.
            distance=cKDTree(predicted).query(actual)[0] if len(predicted) else np.full(len(actual),np.inf)
            finite=distance[np.isfinite(distance)]
            report.append(dict(frame=t,predictions=n,annotated=len(actual),nearest_distance_voxels_median=float(np.median(finite)) if len(finite) else None))
    summary=dict(dataset=url,archive_sha256=digest,train_sequence='01',heldout_sequence='02',
                 train_frames=len(train_ids),heldout_frames=len(test_ids),optimizer_steps=updates,
                 supervision='silver segmentation pseudo-labels',heldout_diagnostic=report)
    (out/'summary.json').write_text(json.dumps(summary,indent=2))
